<a href="https://colab.research.google.com/github/Tulin206/erumhub_deep-learning_2026/blob/main/code/03_Simple_NN_Iris_Classification_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

####  🧠 A Neural Network for the Iris Dataset

This is a **Multi-Class Classification** exercise where we will build a Neural Network from scratch using **only NumPy** to classify the famous **Iris Dataset** into 3 flower species (Setosa, Versicolor, Virginica).

**Key Learning Goals:**
1. Understand **one-hot encoding** (converting labels like 0,1,2 into vectors)
2. Learn **Softmax activation** (for multi-class output probabilities)
3. Learn **Cross-Entropy Loss** (the right loss function for classification)
4. Implement **forward propagation** through 2 layers
5. Implement **backpropagation** with proper gradient calculations
6. See how neural networks learn by adjusting **weights** and **biases**


In [ ]:
import numpy as np
from sklearn.datasets import load_iris

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

#### Step 1: Load and Preprocess Data

Neural networks work best when input data is small (usually between 0 and 1) and centered.

**HINT about ONE-HOT ENCODING:**
- Regular labels: 0, 1, 2 (which species?)
- One-hot format: [1,0,0], [0,1,0], [0,0,1] (probability-like vectors)
- Why? Softmax outputs 3 probabilities (one per class), so labels must be 3 numbers too
- Example: Class 0 (Setosa) → [1,0,0], Class 1 (Versicolor) → [0,1,0], Class 2 (Virginica) → [0,0,1]

**HINT about DATA SCALING (min-max normalization):**
- Raw Iris features: Sepal length 4.3-7.9, Petal width 0.1-2.5 (very different ranges!)
- Neural networks learn FASTER when all inputs are 0-1 (smaller, more uniform)
- Formula: scaled_value = (x - min) / (max - min)
- This converts any feature to 0.0-1.0 range
- Example: if Sepal length ranges 4.3-7.9, then 5.0 → (5.0-4.3)/(7.9-4.3) = 0.163

We need two manual preprocessing functions:
1. **`to_one_hot`**: Converts labels `0` → `[1, 0, 0]`.
2. **`min_max_scale`**: Squashes all input features to be between 0 and 1.


In [ ]:
# Load Data
iris = load_iris()
X_raw = iris.data      # (150 samples, 4 features)
y_raw = iris.target    # (150 samples,) containing 0, 1, 2


In [ ]:
# Define Helper Functions

def to_one_hot(y, num_classes):
    """Converts (150,) array of ints to (150, 3) one-hot matrix"""
    one_hot = np.zeros((y.shape[0], num_classes))
    for i, label in enumerate(y):
        one_hot[i, label] = 1.0
    return one_hot

def min_max_scale(X):
    """Scales features to be between 0 and 1"""
    min_val = X.min(axis=0)
    max_val = X.max(axis=0)
    return (X - min_val) / (max_val - min_val)

In [ ]:
# Preprocess Data
# HINT: Use min_max_scale() function to scale X_raw to [0, 1]
X = #yourcode         # Scale features to [0, 1]

# One-hot encode the outputs
# HINT: num_classes = 3 (we have 3 Iris species)
num_classes = #yourcode
# HINT: Use to_one_hot() function to convert y_raw (0,1,2) to one-hot format
y = #yourcode

In [ ]:
# Shuffle the data (important because Iris is sorted by class!)
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

print("First 5 X samples (scaled):\n", X[:5])
print("\nFirst 5 y samples (one-hot):\n", y[:5])

#### Step 2: Advanced Activation & Loss

**HINT about SOFTMAX (Output Layer):**
- Binary classification: We used Sigmoid to get 1 probability
- Multi-class classification: We use Softmax to get 3 probabilities (one per class)
- Sigmoid: squashes INDEPENDENTLY between 0-1, each neuron separate
- Softmax: squashes probabilities so they ALL SUM to 1.0 together
- Formula: Softmax(z_i) = e^(z_i) / Σ(e^(z_j))
- Example output: [0.7, 0.2, 0.1] means 70% class 0, 20% class 1, 10% class 2
- WHY? Because for classification, probabilities should represent "which one is most likely?" and must sum to 1

**HINT about CROSS-ENTROPY LOSS (Better than MSE for classification):**
- MSE (binary): treated predictions like continuous numbers
- Cross-Entropy (multi-class): understands predictions are PROBABILITIES
- Formula: Loss = -mean(sum(y_true × log(y_pred)))
  - y_true = one-hot labels [1,0,0], [0,1,0], etc.
  - y_pred = softmax probabilities from network
- Why better than MSE? It heavily PENALIZES confident wrong predictions
- Example:
  - If true is [1,0,0] and pred is [0.9,0.05,0.05]: Loss ≈ -log(0.9) ≈ 0.105 (good!)
  - If true is [1,0,0] and pred is [0.1,0.5,0.4]: Loss ≈ -log(0.1) ≈ 2.303 (bad!)

**HINT about THE MAGIC SHORTCUT (Softmax + Cross-Entropy):**
- When you combine Softmax + Cross-Entropy, the gradient is INCREDIBLY simple:
- Gradient = Predicted_Probabilities - True_One_Hot_Labels
- Example: if pred = [0.7, 0.2, 0.1] and true = [1, 0, 0], gradient = [0.7-1, 0.2-0, 0.1-0] = [-0.3, 0.2, 0.1]
- This is why we pair them together in practice! Math simplifies beautifully!


**Sigmoid (For Hidden Layer):**
- Formula: sigmoid(x) = 1 / (1 + e^(-x))
- Takes any number and converts to 0-1
- Derivative: sigmoid'(x) = sigmoid(x) × (1 - sigmoid(x))
- Use sigmoid for hidden layers (non-linear activation makes network powerful)

**Softmax (For Output Layer):**
- Converts logits to probabilities that sum to 1
- Highest logit gets highest probability

In [ ]:
def sigmoid(x):
    # HINT: Same formula as before: 1 / (1 + e^(-x))
    return #yourcode

def sigmoid_derivative(x):
    # HINT: derivative = x * (1 - x) where x is sigmoid output
    return x * (1 - x)

def softmax(x):
    # Subtract max for numerical stability (prevents blowing up exp)
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

#### Step 3: Network Architecture

**HINT about ARCHITECTURE:**
- **Input layer:** 4 neurons (Sepal Length, Sepal Width, Petal Length, Petal Width)
- **Hidden layer:** 6 neurons (adds complexity so network can learn non-linear patterns)
- **Output layer:** 3 neurons (one probability for each Iris species)

**HINT about WEIGHT INITIALIZATION:**
- We use random normal distribution (randn) instead of uniform (rand)
- Multiply by 0.1 to keep weights small initially
- Small weights = network starts with small gradients = stable training
- If weights too big → network might get stuck or diverge

**HINT about HYPERPARAMETERS:**
- **learning_rate:** How big are the steps we take? (usually 0.01-0.1)
- **epochs:** How many times to show all data? (usually 100-1000)


In [ ]:
# HINT: input_neurons = number of features (Sepal Length, Sepal Width, Petal Length, Petal Width)
input_neurons = #yourcode
# HINT: hidden_neurons = you choose (more neurons = more powerful but slower)
hidden_neurons = #yourcode
# HINT: output_neurons = number of classes (3 iris species)
output_neurons = #yourcode

# HINT: learning_rate controls how fast we learn (0.01 = slow/safe, 0.1 = fast/risky)
learning_rate = #yourcode
# HINT: epochs = how many times we show all data to network (more = more training)
epochs = #yourcode

# Initialize weights (standard normal distribution usually works better here)
# We multiply by 0.1 to keep initial weights small

weights_hidden = np.random.randn(input_neurons, hidden_neurons) * 0.1
bias_hidden = np.zeros((1, hidden_neurons))

weights_output = np.random.randn(hidden_neurons, output_neurons) * 0.1
bias_output = np.zeros((1, output_neurons))

#### Step 4: The Training Loop

**HINT about FORWARD PASS:**
- Step 1: hidden_input = X @ weights_hidden + bias_hidden (weighted sum)
- Step 2: hidden_output = sigmoid(hidden_input) (apply activation)
- Step 3: output_input = hidden_output @ weights_output + bias_output (weighted sum)
- Step 4: predicted_output = softmax(output_input) (convert to probabilities)

**HINT about CROSS-ENTROPY LOSS CALCULATION:**
- Formula: Loss = -mean(sum(y_true × log(y_pred)))
- We clip predictions to avoid log(0) errors: use epsilon = 1e-15
- Example: if y_true = [1,0,0] and y_pred = [0.7, 0.2, 0.1]
  - Only first element matters: -log(0.7) ≈ 0.357
  - Average this over all samples

**HINT about BACKWARD PASS (Backpropagation):**
- This is where the "magic shortcut" comes in!
- Step 1: d_predicted_output = (predicted_output - y) / batch_size
  - That's it! Softmax + Cross-Entropy gradient is just difference!
  - Divide by batch_size to normalize
- Step 2: error_hidden = d_predicted_output @ weights_output.T (propagate error backward)
- Step 3: d_hidden_layer = error_hidden * sigmoid_derivative(hidden_layer_output)
  - Multiply by sigmoid derivative (chain rule!)

**HINT about WEIGHT UPDATE:**
- weights_output -= hidden_output.T @ d_predicted_output * learning_rate
  - Move opposite to gradient direction (subtract gradient)
  - Use matrix multiplication @ with learning_rate scale
- bias_output -= sum(d_predicted_output) * learning_rate
  - Sum gradients over all samples
  - Use keepdims=True to keep shape (1, num_classes)


In [ ]:
loss_history = []

for i in range(epochs):
    # 1. FORWARD PASS
    # HINT: Compute weighted sum at hidden layer
    hidden_layer_input = np.dot(X, weights_hidden) + bias_hidden
    # HINT: Apply sigmoid activation to hidden layer
    hidden_layer_output = sigmoid(hidden_layer_input)

    # HINT: Compute weighted sum at output layer
    output_layer_input = np.dot(hidden_layer_output, weights_output) + bias_output
    # HINT: Use SOFTMAX (not sigmoid!) for multi-class probabilities
    predicted_output = softmax(output_layer_input)

    # 2. LOSS (Cross-Entropy for monitoring)
    # Small epsilon to prevent log(0) errors
    epsilon = 1e-15
    clipped_preds = np.clip(predicted_output, epsilon, 1 - epsilon)
    loss = -np.mean(np.sum(y * np.log(clipped_preds), axis=1))
    loss_history.append(loss)

    # 3. BACKWARD PASS
    # HINT: This is the "magic shortcut" - gradient is just (Pred - Actual)
    # HINT: Divide by batch size (X.shape[0]) to normalize
    d_predicted_output = (predicted_output - y) / X.shape[0]

    # HINT: Backprop error to hidden layer
    # HINT: Use matrix multiplication @ with transpose of weights_output
    error_hidden = d_predicted_output.dot(weights_output.T)
    # HINT: Multiply by sigmoid derivative for chain rule
    d_hidden_layer = error_hidden * sigmoid_derivative(hidden_layer_output)

    # 4. UPDATE WEIGHTS
    # HINT: Update output weights: subtract learning_rate × gradient
    weights_output -= hidden_layer_output.T.dot(d_predicted_output) * learning_rate
    # HINT: Update output bias: sum gradients across samples
    bias_output -= np.sum(d_predicted_output, axis=0, keepdims=True) * learning_rate
    # HINT: Update hidden weights: same pattern
    weights_hidden -= X.T.dot(d_hidden_layer) * learning_rate
    # HINT: Update hidden bias: same pattern
    bias_hidden -= np.sum(d_hidden_layer, axis=0, keepdims=True) * learning_rate

    if i % 500 == 0:
        print(f"Epoch {i} Loss: {loss:.4f}")

#### Step 5: Testing Accuracy

**HINT about ARGMAX:**
- Our softmax outputs: [0.7, 0.2, 0.1] (probabilities for each class)
- We need to convert to a single class label: 0 (the class with highest probability)
- np.argmax() finds the index of the maximum value
- Example: np.argmax([0.7, 0.2, 0.1]) = 0

**HINT about ACCURACY:**
- Compare predicted_classes with true_classes
- Count how many are the same
- Divide by total number to get percentage
- Formula: accuracy = sum(pred == true) / total_samples

To check accuracy, we convert our one-hot predictions back into single integers using `np.argmax` (which finds the index of the highest probability).


In [ ]:
# Final forward pass to get predictions
hidden_out = sigmoid(np.dot(X, weights_hidden) + bias_hidden)
final_preds_prob = softmax(np.dot(hidden_out, weights_output) + bias_output)

# Convert probabilities to class labels (0, 1, or 2)
# HINT: np.argmax finds index of highest probability (which class?)
predicted_classes = np.argmax(final_preds_prob, axis=1)
true_classes = np.argmax(y, axis=1)

# Calculate accuracy
# HINT: accuracy = fraction of correct predictions out of total
accuracy = np.mean(predicted_classes == true_classes)
print(f"\nFinal Training Accuracy: {accuracy * 100:.2f}%")

# Show a few examples
print("\nSample Predictions vs True Labels:")
print(f"Predicted: {predicted_classes[:10]}")
print(f"True:      {true_classes[:10]}")